In [1]:
import numpy as np

In [2]:
class localvolDynamics:

    def __init__(self, S0, r, q, maxvol, localvol):
        self.S0 = S0
        self.r = r
        self.q = q
        self.maxvol = maxvol
        self.localvol = localvol


In [3]:
hw3dynamics = localvolDynamics(S0 = 100, r = 0.06, q = 0.01, maxvol = 0.6,
                     localvol = lambda S,t: np.minimum(0.2+5*np.log(S/100)**2+0.1*np.exp(-t), 0.6))

# Note that hw3dynamics.localvol is a function
# that may be invoked in the usual way, for example:
# hw2dynamics.localvol( exchangerate , time )

In [4]:
class CallOnAmericanPut:

    def __init__(self, putexpiry, putstrike, callexpiry, callstrike):
        self.putexpiry = putexpiry
        self.putstrike = putstrike
        self.callexpiry = callexpiry
        self.callstrike = callstrike


In [5]:
hw3contract = CallOnAmericanPut(putexpiry=0.75, putstrike=95, callexpiry=0.25, callstrike=10)

In [6]:
class TreeEngine:

    def __init__(self, N):
        self.N = N

    def price_compound_localvol(self, contract, dynamics):

        S0, r, q = dynamics.S0, dynamics.r, dynamics.q
        localvol = dynamics.localvol
        maxvol = dynamics.maxvol

        T_put = contract.putexpiry
        T_call = contract.callexpiry
        K_put = contract.putstrike
        K_call = contract.callstrike

        N = self.N
        deltat = T_put / N

        # N reject section
        call_step_float = T_call / deltat
        call_step = int(round(call_step_float))
        if abs(call_step_float - call_step) > 1e-8:
            raise ValueError("This N fails to place the call expiry on a tree time point.")

        # Using maxvol as the representative vol for deltax so that Pm >= 0 everywhere.
        sigma_avg = maxvol
        deltax = sigma_avg * np.sqrt(3 * deltat)

        j_terminal = np.arange(N, -N - 1, -1)
        Sgrid = S0 * np.exp(j_terminal * deltax)
        V = np.maximum(K_put - Sgrid, 0.0)

        V_call_snapshot = None

        # Backward induction
        for n in range(N - 1, -1, -1):
            t_n = n * deltat

            j_now = np.arange(n, -n - 1, -1)
            Sgrid_n = S0 * np.exp(j_now * deltax)

            sigma = localvol(Sgrid_n, t_n)
            nu = (r - q) - 0.5 * sigma**2

            Pu = 0.5 * ((sigma**2 * deltat + nu**2 * deltat**2) / (deltax**2) + nu * deltat / deltax)
            Pd = 0.5 * ((sigma**2 * deltat + nu**2 * deltat**2) / (deltax**2) - nu * deltat / deltax)
            Pm = 1 - Pu - Pd # shortcut

            continuation = np.exp(-r * deltat) * (Pu * V[:-2] + Pm * V[1:-1] + Pd * V[2:])

            exercise = np.maximum(K_put - Sgrid_n, 0.0)
            V = np.maximum(continuation, exercise)

            if n == call_step:
                V_call_snapshot = np.maximum(V - K_call, 0.0)

        price_of_put = V[0]

        V = V_call_snapshot
        for n in range(call_step - 1, -1, -1):
            t_n = n * deltat

            j_now = np.arange(n, -n - 1, -1)
            Sgrid_n = S0 * np.exp(j_now * deltax)

            sigma = localvol(Sgrid_n, t_n)
            nu = (r - q) - 0.5 * sigma**2

            Pu = 0.5 * ((sigma**2 * deltat + nu**2 * deltat**2) / (deltax**2) + nu * deltat / deltax)
            Pd = 0.5 * ((sigma**2 * deltat + nu**2 * deltat**2) / (deltax**2) - nu * deltat / deltax)
            Pm = 1 - Pu - Pd

            V = np.exp(-r * deltat) * (Pu * V[:-2] + Pm * V[1:-1] + Pd * V[2:])

        price_of_call_on_put = V[0]

        return (price_of_put, price_of_call_on_put)

In [7]:
hw3tree = TreeEngine(N=30)  #change if necessary to get $0.01 accuracy, in your judgment

In [8]:
(answer_part_a, answer_part_b) = hw3tree.price_compound_localvol(hw3contract,hw3dynamics)

In [9]:
(answer_part_a, answer_part_b)

(np.float64(6.688323263799576), np.float64(1.6399632082505398))